<a href="https://colab.research.google.com/github/jsansao/teic-20231/blob/main/TEIC_Licao27bis_B2W_csv_sentiment_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Colab: Fine-Tuning do BERT (Português) para Análise de Sentimentos (B2W-Reviews)

Este notebook demonstra o processo de fine-tuning de um modelo BERT pré-treinado para a língua portuguesa (`neuralmind/bert-base-portuguese-cased`) para a tarefa de classificação de sentimentos.

Usaremos o dataset `b2w-reviews`, que contém reviews de produtos em português.

## Passo 1: Instalação e Configuração

Primeiro, instalamos as bibliotecas necessárias.

In [1]:
!pip install transformers datasets evaluate

In [2]:
import string
import pandas as pd
# Para pré-processamento
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import re
from sklearn.model_selection import train_test_split


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
# Importações principais
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

In [4]:
!git clone https://github.com/americanas-tech/b2w-reviews01.git

Cloning into 'b2w-reviews01'...
remote: Enumerating objects: 10, done.
remote: Total 10 (delta 0), reused 0 (delta 0), pack-reused 10 (from 1)
Receiving objects: 100% (10/10), 19.82 MiB | 14.89 MiB/s, done.


In [5]:
# Célula 2: Carregar Dados da B2W-Reviews de uma URL

def carregar_dados_b2w():
    """Carrega o dataset B2W-Reviews a partir de uma URL e trata os labels."""

    # URL de um Gist com o dataset (pode levar um momento para carregar)
    url = "b2w-reviews01/B2W-Reviews01.csv"

    print(f"Carregando dados de: {url}")
    df = pd.read_csv(url, sep=',')

    # Remove colunas desnecessárias e linhas com dados faltantes
    df = df[['review_text', 'overall_rating']].dropna()

    # --- Mapeamento de Labels (Rating -> Sentimento) ---
    # 1. Remove reviews neutras (rating == 3)
    df_filtered = df.query("overall_rating != 3").copy()

    # 2. Mapeia 1-2 estrelas para 0 (Negativo) e 4-5 estrelas para 1 (Positivo)
    df_filtered['sentiment'] = df_filtered['overall_rating'].apply(lambda x: 1 if x > 3 else 0)

    # Seleciona apenas as colunas que vamos usar
    df_final = df_filtered[['review_text', 'sentiment']]

    return df_final

df = carregar_dados_b2w()

print("\nDados carregados e mapeados:")
print(f"Total de amostras: {len(df)}")
print("\nDistribuição de sentimentos:")
print(df['sentiment'].value_counts())

display(df.head())

Carregando dados de: b2w-reviews01/B2W-Reviews01.csv


/tmp/ipykernel_3000/494068140.py:10: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, sep=',')



Dados carregados e mapeados:
Total de amostras: 113088

Distribuição de sentimentos:
sentiment
1    79316
0    33772
Name: count, dtype: int64


,review_text,sentiment
0,Estou contente com a compra entrega rápida o ú...,1
1,"Por apenas R$1994.20,eu consegui comprar esse ...",1
2,SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...,1
3,MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...,1
4,"A entrega foi no prazo, as americanas estão de...",1


## Passo 2: Carregamento do Dataset (B2W-Reviews)

Carregamos o dataset `b2w-reviews`. Este dataset possui as colunas `review_text` (o texto) e `stars` (de 1 a 5).

In [6]:
# Célula 3: Função de Pré-processamento (Português)

# Obtém a lista de stopwords em PORTUGUÊS
stop_words = set(stopwords.words('portuguese'))

def preprocessar_texto(text):
    """
    Limpa o texto:
    1. Remove tags HTML (embora esta base não tenha muitas)
    2. Remove pontuação
    3. Converte para minúsculas
    4. Remove stopwords (em português)
    """
    if not isinstance(text, str):
        return ""

    # 1. Remove HTML
    text = re.sub(r'<[^>]+>', ' ', text)

    # 2. Remove pontuação
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Converte para minúsculas
    text = text.lower()

    # 4. Remove stopwords
    palavras = [palavra for palavra in text.split() if palavra not in stop_words]

    return ' '.join(palavras)

# Aplica o pré-processamento
# (Pode levar alguns minutos)
print("Iniciando pré-processamento...")
df['clean_text'] = df['review_text'].apply(preprocessar_texto)
print("Pré-processamento concluído.")

display(df.head())

Iniciando pré-processamento...
Pré-processamento concluído.


,review_text,sentiment,clean_text
0,Estou contente com a compra entrega rápida o ú...,1,contente compra entrega rápida único problema ...
1,"Por apenas R$1994.20,eu consegui comprar esse ...",1,apenas r199420eu consegui comprar lindo copo a...
2,SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...,1,supera agilidade praticidade outras panelas el...
3,MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...,1,filho amou parece verdade tantos detalhes têm
4,"A entrega foi no prazo, as americanas estão de...",1,entrega prazo americanas parabéns smart tv boa...


In [7]:
# Célula 4: Divisão de Treino e Teste


# Como este dataset não tem um split padrão, vamos criar um
# Usaremos 75% para treino e 25% para teste

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Tamanho do treino: {len(X_train)} amostras")
print(f"Tamanho do teste: {len(X_test)} amostras")

Tamanho do treino: 84816 amostras
Tamanho do teste: 28272 amostras


In [8]:
from datasets import Dataset

# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_train.tolist(), 'label': y_train.tolist()}))
test_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_test.tolist(), 'label': y_test.tolist()}))

print("Hugging Face Datasets criados:")
print(train_dataset)
print(test_dataset)

Hugging Face Datasets criados:
Dataset({
    features: ['text', 'label'],
    num_rows: 84816
})
Dataset({
    features: ['text', 'label'],
    num_rows: 28272
})


## Passo 5: Carregamento do Tokenizador e Modelo (Português)

Agora, usamos o `neuralmind/bert-base-portuguese-cased` como nosso modelo base.

* **`cased`**: Significa que o modelo diferencia maiúsculas de minúsculas.
* **`id2label`**: Mapeia os IDs das labels (0 e 1) para nomes legíveis ("Negativo", "Positivo"). Isso melhora a saída do `pipeline`.

In [9]:
# Checkpoint do modelo em português
model_checkpoint = "neuralmind/bert-base-portuguese-cased"

# Carrega o tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)



# Carrega o modelo de classificação
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2
)

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

In [10]:
# --- INÍCIO DA MODIFICAÇÃO (CONGELAMENTO) ---

# 1. Congela todos os parâmetros do corpo do BERT
# (O nome 'bert' é específico para modelos BERT.
# Para RoBERTa seria 'roberta', para DistilBERT seria 'distilbert')
for param in model.bert.parameters():
    param.requires_grad = True

# 2. (Opcional, mas boa prática)
# Garantir que os parâmetros da cabeça de classificação ESTÃO treináveis.
# Eles já vêm com requires_grad=True por padrão, pois são novos.
for param in model.classifier.parameters():
    param.requires_grad = True

# --- FIM DA MODIFICAÇÃO ---

## Passo 6: Pré-processamento (Tokenização)

Esta função aplica o tokenizador aos nossos textos.

In [11]:
def tokenize_function(examples):
    # 'truncation=True' corta textos maiores que o limite máximo do modelo (512 para o BERT)
    # Explicitly add padding here when batched=True to ensure consistent shapes for the datasets library
    return tokenizer(examples["text"], truncation=True, max_length=512)

# Filter out examples with None values in 'text' before mapping
train_dataset_filtered = train_dataset.filter(lambda example: example["text"] is not None)
test_dataset_filtered = test_dataset.filter(lambda example: example["text"] is not None)

# Apply tokenization to the filtered datasets
tokenized_train_dataset = train_dataset_filtered.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset_filtered.map(tokenize_function, batched=True)


# The DataCollator will handle padding dynamically (more efficient)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Filter:   0%|          | 0/84816 [00:00<?, ? examples/s]

Filter:   0%|          | 0/28272 [00:00<?, ? examples/s]

Map:   0%|          | 0/84816 [00:00<?, ? examples/s]

Map:   0%|          | 0/28272 [00:00<?, ? examples/s]

## Passo 7: Definição da Métrica de Avaliação

Usaremos a Acurácia como métrica principal.

In [12]:
# Carrega a métrica de acurácia
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

## Passo 8: Configuração do Treinamento (Fine-Tuning)

Configuramos os `TrainingArguments` e o `Trainer`.

In [13]:
# Define os argumentos do treinamento
training_args = TrainingArguments(
    output_dir="./bert-b2w-finetuned",          # Onde salvar o modelo
    learning_rate=2e-5,                        # Taxa de aprendizado
    per_device_train_batch_size=8,             # Tamanho do batch de treino reduzido
    per_device_eval_batch_size=8,              # Tamanho do batch de avaliação reduzido
    num_train_epochs=3,                        # Número de épocas
    weight_decay=0.01,
    eval_strategy="epoch",               # Avaliar a cada época
    save_strategy="epoch",                     # Salvar a cada época
    load_best_model_at_end=True,               # Carregar o melhor modelo no final
    push_to_hub=False,
    # Adicionar gradient_accumulation_steps se reduzir o batch size não for suficiente
    gradient_accumulation_steps=4, # Example: Accumulate gradients over 2 steps
)

# Cria o objeto Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Passo 9: Iniciar o Treinamento

**Certifique-se de estar em um ambiente com GPU!** (Vá em *Ambiente de execução > Alterar tipo de ambiente de execução > Acelerador de hardware > T4 GPU*).

In [14]:
# Inicia o fine-tuning!
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.559753,0.130177,0.952709
2,0.420297,0.159545,0.956317
3,0.342355,0.194691,0.956353


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7953, training_loss=0.4701083334499855, metrics={'train_runtime': 3061.967, 'train_samples_per_second': 83.1, 'train_steps_per_second': 2.597, 'total_flos': 6438150746918880.0, 'train_loss': 0.4701083334499855, 'epoch': 3.0})

## Passo 10: Avaliação Final

Vejamos a acurácia final do nosso melhor modelo no conjunto de teste.

In [15]:
# Avalia o melhor modelo no dataset de teste
eval_results = trainer.evaluate()

print("\nResultados da Avaliação Final:")
print(eval_results)

Training Loss,Validation Loss,Epoch,Accuracy
0.342355,0.130177,3,0.952709



Resultados da Avaliação Final:
{'eval_loss': 0.13017654418945312, 'eval_accuracy': 0.9527093944538766}


## Passo 11: Teste com Novas Frases (Inferência)

Vamos usar a ferramenta `pipeline` para testar nosso modelo recém-treinado com frases em português.

In [17]:
from transformers import pipeline

# Detecta se há GPU disponível
device = 0 if torch.cuda.is_available() else -1

# Carrega o pipeline de análise de sentimentos com nosso melhor modelo salvo
best_model_path = trainer.state.best_model_checkpoint
print(f"\nCarregando o melhor modelo salvo de: {best_model_path}")

sentiment_pipeline_pt = pipeline(
    "sentiment-analysis",
    model=best_model_path,
    tokenizer=tokenizer,
    device=device  # -1 para CPU, 0 para GPU
)

print("\n--- Teste de Inferência em Português ---")

review_pos = "Este produto é maravilhoso! Superou minhas expectativas, a entrega foi rápida e o material é de ótima qualidade."
review_neg = "Péssimo produto. Não compre."
review_neu = "É um celular ok, funciona, mas a bateria não dura muito. Esperava mais pelo preço."

print(f"\nFrase: {review_pos}")
print(f"Resultado: {sentiment_pipeline_pt(review_pos)}")

print(f"\nFrase: {review_neg}")
print(f"Resultado: {sentiment_pipeline_pt(review_neg)}")

print(f"\nFrase: {review_neu}")
print(f"Resultado: {sentiment_pipeline_pt(review_neu)}")


Carregando o melhor modelo salvo de: ./bert-b2w-finetuned/checkpoint-2651


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Teste de Inferência em Português ---

Frase: Este produto é maravilhoso! Superou minhas expectativas, a entrega foi rápida e o material é de ótima qualidade.
Resultado: [{'label': 'LABEL_1', 'score': 0.9979158043861389}]

Frase: Péssimo produto. Não compre.
Resultado: [{'label': 'LABEL_0', 'score': 0.9958414435386658}]

Frase: É um celular ok, funciona, mas a bateria não dura muito. Esperava mais pelo preço.
Resultado: [{'label': 'LABEL_0', 'score': 0.9288166165351868}]
